# Handwritten Letters and Digits

Can a computer read your handwriting? In this notebook you draw a letter or a digit with your mouse or finger, and a small neural network tries to work out which character it is. It guesses again with every stroke you add, so you can watch it change its mind as you draw.

The network was trained on **EMNIST** (Extended MNIST), a collection of handwritten letters and digits gathered from thousands of different writers. It follows the same format as MNIST, the famous digits-only dataset used in [[82AE4382-A4A5-4F18-BEA8-7513CE9C418A|Handwritten Digits]], but letters make the job much harder: there are more characters to choose from, and some of them look almost the same.

## Loading the model

The trained network is attached to this notebook as a file called `emnist.onnx` (click the paperclip icon in the top right to see it). ONNX stands for Open Neural Network eXchange, a popular format for sharing trained models. The file is about 200 KB, which is still smaller than most photos.

We load it with the `dnn` (deep neural network) module in OpenCV.

In [ ]:
import cv2

net = cv2.dnn.readNetFromONNX('/attachments/emnist.onnx')
print("Neural network loaded")

## The characters the model knows

The model was trained on the **Balanced** version of EMNIST, which has 131,600 images sorted into **47** classes. A class is one of the answers the model can give.

Why 47, and not 62 (10 digits, 26 uppercase letters and 26 lowercase letters)? Because for 15 letters, such as **c**, **o** and **s**, the lowercase letter is just a smaller copy of the uppercase one. Nobody can tell them apart from a single character, so EMNIST puts both in one class. Only the 11 lowercase letters that have their own shape (**a b d e f g h n q r t**) get a class of their own.

The model answers with a class number from 0 to 46. `LABELS` turns that number into a character.

In [ ]:
LABELS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabdefghnqrt"

print(len(LABELS), "classes")
print("Class 10 is", LABELS[10])
print("Class 36 is", LABELS[36])

## Getting your drawing ready for the model

The model can't look at your drawing directly. Every image in EMNIST is a tiny **28 x 28** grid of pixels, with a white character on a black background, so that is the only kind of picture the model understands.

EMNIST characters were also tidied up in the same way: each one was cut out of the form it was written on, placed in the middle of a square with a small margin around it, and then shrunk to 28 x 28. The `prepare` function below does the same to your drawing. That is why it doesn't matter whether you draw big or small, or in a corner of the pad.

In [ ]:
import numpy as np

def prepare(drawing):
  """Turn the drawing (a 2D array, 0 = empty, 255 = ink) into the 28x28
  image the model expects. Returns None if nothing has been drawn."""
  # 1. Find the part of the pad that has ink on it, and cut it out.
  rows, columns = np.nonzero(drawing)
  if len(rows) == 0:
    return None
  character = drawing[rows.min():rows.max() + 1, columns.min():columns.max() + 1]

  # 2. Place it in the middle of a black square, with a small margin
  #    around it, like the EMNIST characters.
  height, width = character.shape
  size = max(height, width)
  margin = size // 12
  square = np.zeros((size + 2 * margin, size + 2 * margin), drawing.dtype)
  top = margin + (size - height) // 2
  left = margin + (size - width) // 2
  square[top:top + height, left:left + width] = character

  # 3. Shrink the square to 28x28.
  return cv2.resize(square, (28, 28), interpolation=cv2.INTER_AREA)

print("prepare function created")

## Asking the model

Now for the neural network itself. We hand it the 28 x 28 image, and it gives back 47 scores, one for each class. The higher a score, the more the image looks like that character to the model.

Raw scores are hard to read, so the last two lines turn them into **confidences** that add up to 100%. This step is called *softmax*.

In [ ]:
def predict(image):
  """Run the model on a 28x28 image. Returns 47 confidences (0 to 1),
  one for each character in LABELS."""
  blob = cv2.dnn.blobFromImage(image, scalefactor=1 / 255.0)
  net.setInput(blob)
  scores = net.forward()[0]
  exponents = np.exp(scores - scores.max())
  return exponents / exponents.sum()

print("predict function created")

In [ ]:
PAD = (10, 10, 320, 320)          # where you draw: x, y, width, height
CLEAR_BUTTON = (10, 340, 320, 30)
INK = "#201E1D"
GREEN = "#56BC6C"
GREY = "#C9C5C1"

def show(canvas, image, confidences):
  """Redraw the pad's background and everything to the right of it."""
  ctx = canvas.get_context("2d")
  ctx.clear_rect(0, 0, 600, 380)
  ctx.text_align = "center"

  # The pad and the Clear button. Your strokes sit on a layer above this.
  ctx.fill_style = INK
  ctx.fill_rect(*PAD)
  ctx.fill_rect(*CLEAR_BUTTON)
  ctx.fill_style = "#ffffff"
  ctx.font = "bold 14px sans-serif"
  ctx.fill_text("Clear", 170, 360)
  if image is None:
    ctx.fill_style = "#8a8580"
    ctx.font = "18px sans-serif"
    ctx.fill_text("Draw a letter or digit here", 170, 175)

  # What the model sees: the 28x28 image, each pixel drawn as a 5x5 block.
  ctx.fill_style = INK
  ctx.font = "12px sans-serif"
  ctx.text_align = "left"
  ctx.fill_text("What the model sees", 350, 22)
  ctx.fill_text("Prediction", 510, 22)
  ctx.fill_text("Top 5 guesses", 350, 196)
  canvas.draw_pixels(np.zeros((28, 28)) if image is None else image, 350, 30, 140, 140)

  # The five most likely characters, best first. The first is the prediction.
  top = [] if confidences is None else [int(i) for i in confidences.argsort()[::-1][:5]]
  ctx.font = "bold 96px sans-serif"
  ctx.text_align = "center"
  ctx.fill_text(LABELS[top[0]] if top else "?", 548, 135)

  # One bar per guess.
  for row, label in enumerate(top):
    y = 206 + row * 33
    confidence = float(confidences[label])
    ctx.fill_style = INK
    ctx.font = "bold 20px sans-serif"
    ctx.text_align = "left"
    ctx.fill_text(LABELS[label], 350, y + 20)
    ctx.font = "13px sans-serif"
    ctx.text_align = "right"
    ctx.fill_text(f"{confidence:.0%}", 412, y + 18)
    ctx.fill_style = GREEN if row == 0 else GREY
    ctx.fill_rect(420, y + 5, 170 * confidence, 16)

def is_clear_button(x, y):
  left, top, width, height = CLEAR_BUTTON
  return left <= x <= left + width and top <= y <= top + height

## Draw a character!

Time to put it all together. `canvas.on_draw` lets you draw on the pad, and calls our `update` function every time your drawing changes. Each update follows the same three steps: **prepare** the drawing, **predict** the character, and **show** the results.

With 47 classes there isn't room for a bar each, so the canvas shows the model's **top 5 guesses**. Keep an eye on the runners-up: they tell you which characters the model is torn between.

Run the cell, then draw a letter or a digit on the black pad. Click **Clear** to start again, and press the Stop button when you are done.

In [ ]:
from codetto import graphics2d

canvas = graphics2d.canvas(600, 380, scale=True)

def update():
  image = prepare(canvas.get_drawing())
  confidences = None if image is None else predict(image)
  show(canvas, image, confidences)
  # The show function is hidden - edit the notebook if you'd like to see it

def handle_click(x, y):
  if is_clear_button(x, y):
    canvas.clear_drawing()
    update()

canvas.on_draw(update, line_width=22, color="#ffffff", area=PAD)
canvas.on_click(handle_click)

update()
canvas.run()

## Things to try

- Draw an **F**, then add a stroke along the bottom to turn it into an **E**. Watch the guesses change as you draw.
- Draw a single vertical line. Is it the digit **1**, the letter **I**, or a lowercase **l**? The model can't be sure either, so its confidence is shared between a few guesses.
- Draw a circle. The letter **O** and the digit **0** are in the top guesses together. How would *you* tell them apart without seeing the rest of the word?
- Draw an uppercase **A**, clear the pad, and draw a lowercase **a**. They have different shapes, so the model has a class for each. Now try a lowercase **s**. Why does the model always answer with the uppercase letter?
- Draw a small character in one corner of the pad. The model still sees it large and centered. Which function is responsible for that?
- Draw something that isn't a letter or digit, like a star or a smiley face. The model always picks one of its 47 characters, because they are all it knows. How confident is it?
- In the last code cell, change `line_width=22` to `line_width=6` and run it again. Thin strokes almost disappear when they are shrunk to 28 x 28. Does the model get worse?

# Check Your Understanding